# 05 — Model Improvements

The v1 model (from notebook 04) never predicts above ~6 points. Here we diagnose why and fix it.

**Problem:** 62% of training rows are zero-minute appearances (player didn't play), dragging predictions toward zero.

**Fix:** Filter training data to only include gameweeks where the player actually played (minutes > 0). This shifts the target distribution from mean ~1.2 to something more representative of actual on-pitch output.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

plt.style.use('ggplot')
pd.set_option('display.max_columns', 40)

## 1. Load fresh data and rebuild features

Using `build_features()` from the prediction pipeline to get the full gameweek DataFrame with raw columns intact.

In [ ]:
from src.fpl.predict import build_features

gw_df = build_features()
print(f'Total gameweek records: {len(gw_df):,}')
print(f'Unique players: {gw_df["player_id"].nunique()}')
print(f'Gameweek range: {gw_df["round"].min()} - {gw_df["round"].max()}')

## 2. The problem: target distribution

Let's see why the v1 model can't predict above ~6 points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
target = gw_df['target']
axes[0].hist(target, bins=range(-2, 25), edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Target Distribution — All Records')
axes[0].set_xlabel('Gameweek Points')
axes[0].set_ylabel('Count')
axes[0].axvline(target.mean(), color='red', linestyle='--', label=f'Mean: {target.mean():.2f}')
axes[0].legend()

# Breakdown
zeros = (target == 0).sum()
played = (gw_df['minutes'] > 0).sum()
print(f'Zero-point records: {zeros:,} ({zeros/len(target)*100:.1f}%)')
print(f'Records where player played (min > 0): {played:,} ({played/len(target)*100:.1f}%)')
print(f'Records where player did NOT play: {len(target)-played:,} ({(len(target)-played)/len(target)*100:.1f}%)')

# Distribution for played-only
played_target = target[gw_df['minutes'] > 0]
axes[1].hist(played_target, bins=range(-2, 25), edgecolor='black', alpha=0.7, color='forestgreen')
axes[1].set_title('Target Distribution — Minutes > 0 Only')
axes[1].set_xlabel('Gameweek Points')
axes[1].set_ylabel('Count')
axes[1].axvline(played_target.mean(), color='red', linestyle='--', label=f'Mean: {played_target.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nMean points (all): {target.mean():.2f}')
print(f'Mean points (played only): {played_target.mean():.2f}')
print(f'Median points (played only): {played_target.median():.1f}')
print(f'95th percentile (played only): {played_target.quantile(0.95):.1f}')

## 3. Prepare filtered feature matrix

Drop rows where the player had 0 minutes, then apply the same time-based split.

In [ ]:
# Feature columns (same as notebook 04)
id_cols = ['player_id', 'web_name', 'position', 'team_name', 'round']
target_col = 'target'
exclude = id_cols + [target_col, 'minutes', 'total_points', 'value', 'kickoff_time',
                     'id', 'code', 'element_type_y', 'team', 'price_prev', 'prev_kickoff']

# Use the same feature columns as notebook 04 (load from saved metadata)
with open(project_root / 'models' / 'model_metadata.json') as f:
    old_metadata = json.load(f)

feature_cols = old_metadata['feature_columns']
print(f'Features: {len(feature_cols)}')

# Filter: only rows where player actually played
df_all = gw_df.dropna(subset=feature_cols + [target_col]).copy()
df_played = df_all[df_all['minutes'] > 0].copy()

print(f'All records (with features): {len(df_all):,}')
print(f'Played records (min > 0):    {len(df_played):,} ({len(df_played)/len(df_all)*100:.1f}%)')

In [ ]:
# Time-based split — same cutoff as notebook 04
TRAIN_END_GW = 22

# V1: all data (reproducing notebook 04)
train_v1 = df_all[df_all['round'] <= TRAIN_END_GW]
test_v1 = df_all[df_all['round'] > TRAIN_END_GW]

# V2: played-only data
train_v2 = df_played[df_played['round'] <= TRAIN_END_GW]
test_v2 = df_played[df_played['round'] > TRAIN_END_GW]

print('=== V1 (all data) ===')
print(f'Train: {len(train_v1):,} rows, target mean: {train_v1[target_col].mean():.2f}')
print(f'Test:  {len(test_v1):,} rows, target mean: {test_v1[target_col].mean():.2f}')

print(f'\n=== V2 (played only) ===')
print(f'Train: {len(train_v2):,} rows, target mean: {train_v2[target_col].mean():.2f}')
print(f'Test:  {len(test_v2):,} rows, target mean: {test_v2[target_col].mean():.2f}')

## 4. Train models on both datasets

Same three models, same hyperparameters — only the training data changes.

In [ ]:
def train_and_evaluate(train_df, test_df, feature_cols, target_col, label):
    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]

    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(
            n_estimators=200, max_depth=10, min_samples_leaf=20,
            random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1
        ),
    }

    results = {}
    trained = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        trained[name] = model
        y_pred = model.predict(X_test)
        results[name] = {
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': root_mean_squared_error(y_test, y_pred),
            'R\u00b2': r2_score(y_test, y_pred),
        }

    results_df = pd.DataFrame(results).T.round(3)
    print(f'\n=== {label} ===')
    print(results_df.to_string())
    return results_df, trained

print('Training on V1 (all data)...')
results_v1, models_v1 = train_and_evaluate(train_v1, test_v1, feature_cols, target_col, 'V1: All Data')

print('\nTraining on V2 (played only)...')
results_v2, models_v2 = train_and_evaluate(train_v2, test_v2, feature_cols, target_col, 'V2: Played Only')

## 5. Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['MAE', 'RMSE', 'R\u00b2']
colors_v1 = ['#6c8ebf', '#6c8ebf', '#6c8ebf']
colors_v2 = ['#82b366', '#82b366', '#82b366']

for i, metric in enumerate(metrics):
    x = np.arange(3)
    width = 0.35
    axes[i].bar(x - width/2, results_v1[metric], width, label='V1 (all data)', color='steelblue', alpha=0.8)
    axes[i].bar(x + width/2, results_v2[metric], width, label='V2 (played only)', color='forestgreen', alpha=0.8)
    axes[i].set_title(metric)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(['LR', 'RF', 'XGB'], rotation=0)
    axes[i].legend(fontsize=8)

plt.suptitle('V1 vs V2 Model Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

# Improvement summary
print('=== Improvement (V2 vs V1) ===')
for model_name in results_v1.index:
    mae_diff = results_v1.loc[model_name, 'MAE'] - results_v2.loc[model_name, 'MAE']
    r2_diff = results_v2.loc[model_name, 'R\u00b2'] - results_v1.loc[model_name, 'R\u00b2']
    print(f'{model_name:20s}  MAE: {mae_diff:+.3f}  R\u00b2: {r2_diff:+.3f}')

## 6. Prediction distribution comparison

The key question: does V2 produce more realistic prediction ranges?

In [ ]:
best_name_v2 = results_v2['MAE'].idxmin()
best_v1 = models_v1[best_name_v2]
best_v2 = models_v2[best_name_v2]

preds_v1 = best_v1.predict(test_v1[feature_cols])
preds_v2 = best_v2.predict(test_v2[feature_cols])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(preds_v1, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title(f'V1 Predictions ({best_name_v2})')
axes[0].set_xlabel('Predicted Points')
axes[0].axvline(preds_v1.mean(), color='red', linestyle='--', label=f'Mean: {preds_v1.mean():.2f}')
axes[0].axvline(preds_v1.max(), color='orange', linestyle='--', label=f'Max: {preds_v1.max():.2f}')
axes[0].legend()

axes[1].hist(preds_v2, bins=40, edgecolor='black', alpha=0.7, color='forestgreen')
axes[1].set_title(f'V2 Predictions ({best_name_v2})')
axes[1].set_xlabel('Predicted Points')
axes[1].axvline(preds_v2.mean(), color='red', linestyle='--', label=f'Mean: {preds_v2.mean():.2f}')
axes[1].axvline(preds_v2.max(), color='orange', linestyle='--', label=f'Max: {preds_v2.max():.2f}')
axes[1].legend()

plt.suptitle('Prediction Range: V1 vs V2', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'V1 prediction range: {preds_v1.min():.2f} to {preds_v1.max():.2f} (mean {preds_v1.mean():.2f})')
print(f'V2 prediction range: {preds_v2.min():.2f} to {preds_v2.max():.2f} (mean {preds_v2.mean():.2f})')
print(f'\nV1 predictions > 5: {(preds_v1 > 5).sum()}')
print(f'V2 predictions > 5: {(preds_v2 > 5).sum()}')

## 7. Feature importance (V2 best model)

In [ ]:
best_model_v2 = models_v2[best_name_v2]

if hasattr(best_model_v2, 'feature_importances_'):
    importances = best_model_v2.feature_importances_
elif hasattr(best_model_v2, 'coef_'):
    importances = np.abs(best_model_v2.coef_)

feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
feat_imp.tail(20).plot(kind='barh', ax=ax, color='forestgreen')
ax.set_title(f'Top 20 Feature Importances — V2 {best_name_v2}')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 10 features:')
for feat, imp in feat_imp.tail(10).iloc[::-1].items():
    print(f'  {feat:30s}  {imp:.4f}')

## 8. Top player predictions — sanity check

Let's see what V2 predicts for well-known players.

In [ ]:
# Generate predictions for all players using latest gameweek data
latest = gw_df.groupby('player_id').tail(1).copy()
latest = latest.dropna(subset=feature_cols)
latest_played = latest[latest['minutes'] > 0].copy()

latest_played['pred_v2'] = best_model_v2.predict(latest_played[feature_cols])
latest_played['pred_v1'] = best_v1.predict(latest_played[feature_cols])

top_20 = latest_played.nlargest(20, 'pred_v2')
print('=== Top 20 Predicted Players (V2) ===')
print(top_20[['web_name', 'position', 'team_name', 'price', 'pred_v2', 'pred_v1']].to_string(index=False))

## 9. Save improved model

In [ ]:
# XGBoost gives the widest prediction range, making it most useful for FPL.
# While all models have similar MAE (~2.2), XGBoost better differentiates top players.
best_name_v2 = 'XGBoost'
best_model_v2 = models_v2[best_name_v2]

models_dir = project_root / 'models'

# Save V2 model
model_path = models_dir / 'best_model.joblib'
joblib.dump(best_model_v2, model_path)

# Save metadata
metrics_v2 = results_v2.loc[best_name_v2].to_dict()
metadata = {
    'model_name': best_name_v2,
    'version': 'v2',
    'improvement': 'Trained on played-only data (minutes > 0)',
    'feature_columns': feature_cols,
    'train_gw_range': [4, TRAIN_END_GW],
    'test_gw_range': [TRAIN_END_GW + 1, int(gw_df['round'].max())],
    'metrics': metrics_v2,
}
with open(models_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved V2 {best_name_v2} to {model_path}')
print(f'Model size: {model_path.stat().st_size / 1024:.0f} KB')
print(f'Metrics: {metrics_v2}')

## 10. Learnings & Key Takeaways

### Why were predictions capped at ~5 points?

The v1 model (notebook 04) never predicted above ~6 points for any player, even though top FPL assets 
regularly score 10+ in a gameweek. Three root causes:

1. **Sparse target distribution:** 62% of all gameweek records were zero-minute appearances 
   (player not in the squad). These contributed nothing but noise, dragging the learned mean 
   toward zero and compressing the prediction range.

2. **Rolling averages suppress variance:** Using 3 and 5-game rolling averages of stats smooths 
   out the very spikes we want to predict. When most games are 0 points, the rolling average 
   stays close to zero even for good players.

3. **Regression to the mean:** Tree-based models (Random Forest, XGBoost) predict the average 
   outcome for similar feature vectors. With a training mean of ~1.2, predictions cluster there.

### What we fixed

- **Filtered to minutes > 0:** Removed all zero-minute records from training. This shifted the 
  target mean from ~1.2 to ~3.3 and gave the model a more realistic distribution to learn from.
- **Selected XGBoost over Linear Regression:** Despite similar MAE (~2.2), XGBoost produces a 
  wider prediction range (up to 7.4 vs 6.1). For FPL, differentiating top picks matters more 
  than minimizing average error.
- **Filtered inference to active players:** The API now only returns predictions for players who 
  played in their most recent gameweek (306 players vs 817), removing noise from injured/benched players.

### Results: V1 vs V2

| Metric | V1 (all data) | V2 (played only) |
|--------|---------------|-------------------|
| Mean prediction | 1.2 pts | 3.2 pts |
| Max prediction | 5.9 pts | 7.4 pts |
| Predictions > 5 pts | 4 players | 18 players |
| Active players | 817 | 306 |
| Model | Random Forest | XGBoost |

### The fundamental limitation

Per-gameweek FPL points are inherently stochastic. A player scoring 15+ requires specific in-game 
events (hat trick, multiple assists, clean sheet + bonus) that historical rolling averages cannot 
fully predict. The R² remains low (~0.02), meaning the model explains very little of the variance.

This is a known challenge in sports prediction — even bookmakers' models have limited accuracy for 
individual player fantasy points. The model is most useful for **ranking** players (who is likely 
to score MORE) rather than predicting exact point totals.

### Future improvements to explore

- **Opponent defensive strength:** How many goals/xG does the opposing team concede?
- **Set piece takers:** Penalty/corner/free kick takers have higher expected returns
- **Expected minutes model:** Predict probability of starting, then weight predictions
- **Fixture congestion:** Players in cup competitions may be rotated
- **Form momentum:** Weight recent games more heavily than older ones
- **Quantile regression:** Predict the 75th percentile (upside) instead of the mean